# **UPI Transaction Analysis (2021-2022)**

In [ ]:
import pandas as pd

df1 = pd.read_csv("/Users/bhumi/Desktop/pandas project/UPI apps transaction data in 2021.csv")
df2 = pd.read_csv("/Users/bhumi/Desktop/pandas project/UPI apps transaction data in 2022 - - in 2022.csv.csv")

df2.columns = df2.columns.str.strip()

df = pd.concat([df1, df2], ignore_index=True)

df

In [ ]:
df.rename(columns={'Volume (Mn) By Costumers':'Volume (Mn) By Customers',
                  'Value (Cr) by Costumers':'Value (Cr) By Customers' },inplace=True)
print(df.columns)

## **The UPI dataset contains transaction from year 2021-2022**
## It includes the Bank-wise transaction volume across the india and major apps used in India


In [ ]:
print("Data Overview")
df.info()
print("Data Description")
df.describe()

### **Null Checking the Values and changing the datatype of given columns**
#### **No null values are found.Convert the numeric object into int64**


In [ ]:
print("Null Checking")
print(df.isnull().sum())
cols_to_Convert=['Volume (Mn) By Customers','Value (Cr) By Customers','Volume (Mn)','Value (Cr)']
for cols in cols_to_Convert:
    df[cols]=pd.to_numeric(df[cols].astype(str).str.replace(',',''),errors='coerce')
df.dtypes


## **First Five Rows of  dataset**

In [ ]:
print('First Five Rows')
df.head()

## **Last Five Rows of dataset**

In [ ]:
print('Last Five Rows')
df.tail()

### **Data Cleaning**
### Removed whitespace and check the inconsistent name of UPI Banks

In [ ]:
df['UPI Banks']=df['UPI Banks'].str.strip()
print("Unique UPI after strip")
print(df['UPI Banks'].unique())
print('Insight: The white space is removed but some UPI Bank name have inconsistent name ')

## **Data Cleaning the inconsistent UPI banks Names**
#### Using replace function and unqiue keyword to print the original name of UPI Banks
#### After cleaning the total 75 number of UPI Banks are present.

In [ ]:
df['UPI Banks']=df['UPI Banks'].str.replace(r'\s+[Aa]pps?$','',regex=True)
df['UPI Banks']=df['UPI Banks'].replace({
    'Punjab Sind Bank' : 'Punjab Sindh Bank',
    'Janta Sahakari Bank':'Janata Sahakari Bank',
    'WhatsAppp*':'WhatsApp',
    'Jupiter Money':'Jupiter',
    'Jupiter Edge (LivQuick PPI App)':'Jupiter',
    'Jupiter Edge (LivQuick PPI)':'Jupiter',
    'Fino Payments bank':'Fino Payments Bank',
    'Other':'Others',
    'Other Bank':'Others',
    'WhatsApp*':'WhatsApp',
    'Bajaj Markets':'Bajaj Finserv',
    'Finserv Markets':'Bajaj Finserv',
     'GoIbibo':'Goibibo',
    'MakeMy Trip':'MakeMyTrip' })
print('Unique banks after cleaning:',df['UPI Banks'].nunique())
print(df['UPI Banks'].unique())
print('Insight: Here the  unique data of UPI Banks are present ')

### **Checking the total number of times the UPI banks appear in dataset**
#### Insight : 77 banks are total lenght and Bajaj Finserv has the high appearance in the dataset Goibibo and bandhan Bank has ony 1 time apperance

In [ ]:
print('The Count of UPI Banks Present here:')
print(df['UPI Banks'].value_counts())
print('Insight: The total number of UPI Bank account  are present')

### **Each Year Valuation**
#### On year 2021:654 and 2022:452 total transactions are present

In [ ]:
print("How many transactions are present in each year?:")
print(df['Year'].value_counts())
print('Insight: The transactions record distributed over the year')

## **Total Transaction Value by each Year**
#### Total value transaction value groupby by each bank across year 2021-22

In [ ]:
print("The total transaction value in (Cr) for each bank :")
print(df.groupby('UPI Banks')['Value (Cr) By Customers'].sum())
print('Insight: High market concentration in digital payments')

### **Top 10 Bank by transaction values
#### PhonePe dominates the total top transaction by almost 33.6 cr rs.,followed by Google pay and paytm payments banks

In [ ]:
print('Top 10 Highest Transaction Done by each UPI Banks')
print(df.groupby('UPI Banks')['Value (Cr) By Customers'].sum().sort_values(ascending=False).head(10))
print('Insight: The companies like PhonePe,GooglePay and Paytm payments bank are dominating across the UPI transactions')

### **Least Transaction bank by values in Cr**
#### superpay bank and bandhan bank has the lowest adaption for customers of value approx 1.50

In [ ]:
print('Lowest Transaction Made UPI Banks are:')
print(df.groupby('UPI Banks')['Value (Cr) By Customers'].sum().sort_values(ascending=False).tail(10))
print('Insight: The lowest adopted UPI Bank transaction made in bank SuperPay which show the lack of service and production')

## **Bank wise Monthly Transaction Values**
### Monthly transaction value grouped by each bank and month

In [ ]:
print('Bank wise monthly Transaction values')
num_cols=['Value (Cr) By Customers','Volume (Mn) By Customers','Volume (Mn)','Value (Cr)']
df[num_cols]=df[num_cols].apply(pd.to_numeric,errors='coerce')
print(df.groupby(['UPI Banks','Month'])['Value (Cr) By Customers'].sum().reset_index())


## **Yearly Avg transaction value per bank**
### Yes bank shows NaN for 2022 year - no records exists for that year in the dataset

In [ ]:
print('Yearly average transaction value per bank;')
print(df.groupby(['UPI Banks','Year'])['Value (Cr) By Customers'].mean().reset_index())
print('Insight: Yes bank shows NaN for 2022- No record exist for that year in the dataset')

### **Month Name Mapping**
#### Added a 'Month Name' column by mapping numbers to names for better readability.

In [ ]:
month_map={1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
           7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['Month Name']=df['Month'].map(month_map)
df=df.sort_values(['Year','Month']).reset_index(drop=True)
print(df[['Month Name','Month']].head())
print('Insight: Data span monthly showing fro jan 2021')
print('Month Name column is added for better readability')

## **Bank Category Classification**
### Only 5 banks fall into giant category .This 5 Banks has highest transactions in India.

In [ ]:
def bank_category(value):
    if value>=100000:
        return 'Giant'
    elif value>=10000:
        return 'Large'
    elif value>=1000:
        return 'Medium'
    else:
        return 'Small'
bank_totals=df.groupby('UPI Banks')['Value (Cr) By Customers'].sum().reset_index()
bank_totals['Category']=bank_totals['Value (Cr) By Customers'].apply(bank_category)
print(bank_totals.sort_values('Value (Cr) By Customers', ascending=False))
print('Insight: Only 5 Banks fall in Giant category (PhonePe,Google pay,Paytm payments bank) dominating the UPI systems')
print('Majority of banks fall into the Large category showing large concentration on Giant category banks')

### **ADDING MATPLOTLIB LIBRARY**

In [ ]:
import matplotlib.pyplot as plt

## **CHART 1: Top 10 Banks by transaction volume**
#### Top 10 UPI banks ranked by total transactiion volume(cr) across 2021-22 year
#### PhonePe leads by huge volume followed by google pay and paytm payments banks.The gap between the 3 banks are massive

In [ ]:
df['Volume (Mn)']=pd.to_numeric(df['Volume (Mn)'],errors='coerce')
top10=df.groupby('UPI Banks')['Volume (Mn)'].sum().sort_values(ascending=True).tail(10)
plt.figure(figsize=(16,5))
plt.plot(top10.index,top10.values,marker='o',color='red')
plt.title('Top 10 Bank by Volume')
plt.xlabel('Volume (Mn)')
plt.ylabel('UPI Banks')
plt.tight_layout()
plt.show()

## **CHART 2: Monthly UPI Transaction Analysis**
### Total UPI transaction value month wise across year 2021-22
### Transactioon values show increasing trend in starting month at the middle the trend goes down and then become normal


In [ ]:
df['Value (Cr)']=pd.to_numeric(df['Value (Cr)'],errors='coerce')
monthly=df.groupby('Month')['Value (Cr)'].sum()
plt.figure(figsize=(16,5))
plt.plot(monthly.index,monthly.values,marker='o',color='blue')
plt.title("Monthly UPI transaction")
plt.xlabel('Month')
plt.ylabel('Value (Cr)')
plt.tight_layout()
plt.show()

## **Total transaction value 2021 vs 2022**
Year wise total UPI transaction value comparison.
### The value of UPI transaction for year 2022 is only of 7 months so, the total is lower then the 2021 year value transactions.


In [ ]:
df['Value (Cr)']=pd.to_numeric(df['Value (Cr)'].astype(str).str.replace(',',''),errors='coerce')
total_value=df.groupby('Year')['Value (Cr)'].sum()
plt.figure(figsize=(10,5))
plt.bar(['2021','2022'],total_value.values,color=['green','red'],width=0.4)
plt.title('Total Value of Year 2021 vs 2022')
plt.xlabel('Year')
plt.ylabel('Value (Cr)')
plt.tight_layout()
plt.show()
print(df.groupby('Year')['Month'].nunique())